# KWS with iRDT
Code including a novel feature extractor for signal classification
Suppport for article
Radu Dogaru and Ioana Dogaru, "A Multiplication-Free Feature Extractor for Signal Classification: Keyword Spotting Case Study", august 2026.

Copyright Radu and Ioana Dogaru, August 2026
Correspondence to radu.dogaru@upb.ro

In [ ]:
# Basic libraries, cloning git and init working path
!pip install ai_edge_litert
!git clone https://github.com/radu-dogaru/rdt_transform_for_tiny_ml_signal_classifiers/
import os
os.environ["KERAS_BACKEND"] = 'tensorflow'
import keras
from ai_edge_litert.interpreter import Interpreter
import librosa
import math
import numpy as np
from numba import jit
from IPython.display import Audio

#
local_path='/content/rdt_transform_for_tiny_ml_signal_classifiers/Keyword_Spotting/'
os.chdir(local_path)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.3/21.3 MB 33.9 MB/s eta 0:00:00
Cloning into 'rdt_transform_for_tiny_ml_signal_classifiers'...
remote: Enumerating objects: 119, done.
remote: Counting objects: 100% (119/119), done.
remote: Compressing objects: 100% (116/116), done.
remote: Total 119 (delta 54), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (119/119), 6.09 MiB | 20.31 MiB/s, done.
Resolving deltas: 100% (54/54), done.


# The iRDTv feature extractor

In [ ]:
import math
from numba import jit
@jit
def irdtv(signal, M=8, w=256, chan=[1,2,4,8, 16, 32, 64, 100], biti=24, log=False):
    # Copyright Radu DOGARU, radu.dogaru@upb.ro Aug. 1, 2026 -
    # More details about theory here https://github.com/radu-dogaru/rdt_transform_for_tiny_ml_signal_classifiers
    # conversion of [-1,1] to fixed int
    tip='int32'
    multi=(2**(biti-1-math.log2(w)-2)) # multiplier to convert [-1,1] floats to intergers
    signal=(multi*signal).astype(tip)
    # fixed point computation next
    m = len(chan)            # number of delays (a.k.a. filter banks )
    windows=len(signal)//w   # w is usually a power of 2  -> shift
    frames=windows//M  # number of frames for a given M (M=win_per_frame)
    Feat_spec = np.zeros((frames,m)).astype('int32')
    spectrum = np.empty((windows,m)).astype('int32')  # allocate memory for spectrum lines
    #============  first loop ================================================
    limit=w/4 ; base=0
    for i in range(0,windows): # Compute the RDT "spectrum for each window "
        for k in range(0,m):
            delay=chan[k]
            spik=0
            for t in range(limit,w-limit):
                spik+=abs(signal[base+t-delay]+signal[base+t+delay]-2*signal[base+t])  # main computation here
            spectrum[i,k]= spik
        base+=w
    # ========================== second loop ==================================
    # To avoid padding the signal, first and last spectrum lines are ignored (they contain erroneous data)
    base=1
    for k in range(0,frames-1):
        for col in range(0,m):
            F_k_col=0
            for i in range(base,base+M):
                F_k_col += spectrum[i,col]
            Feat_spec[k,col]=F_k_col
        base+=M
    #++++++++++++++++++++++++++++++++++++  back to float
    Feat_spec=Feat_spec/(multi*1.0)
    if log:
      Feat_spec=np.log2(1+Feat_spec)
    return frames, Feat_spec

# Make a folder with 20 samples from each of the 12 classes
- note: one may load their own .wav with commands from the same classes
(some are already included for testing)

In [ ]:
import zipfile

# Path to folder with signals from the original dataset
nume_fisier_zip = 'date_demo.zip'
folder_destinatie = 'KWS'

# extract it
with zipfile.ZipFile(nume_fisier_zip, 'r') as zip_ref:
    zip_ref.extractall(folder_destinatie)
    print(f"Arhchive unpacked in folder: '{folder_destinatie}'")

Arhchive unpacked in folder: 'KWS'


In [ ]:
import os
import random
# utility for random extraction of a sample
def extrage_fisier_aleator(folder_principal):
    # Obținem lista tuturor subfolderelor din folderul principal
    try:
        subfoldere = [f for f in os.listdir(folder_principal)
                      if os.path.isdir(os.path.join(folder_principal, f))]
    except FileNotFoundError:
        print(f"Eroare: Folderul '{folder_principal}' nu a fost găsit.")
        return None, None

    if not subfoldere:
        print("Eroare: Nu s-a găsit niciun subfolder în directorul specificat.")
        return None, None

    # 1. Alegem în mod aleator unul dintre cele 12 subfoldere (aceasta este eticheta)
    eticheta_aleasa = random.choice(subfoldere)
    cale_subfolder = os.path.join(folder_principal, eticheta_aleasa)

    # Obținem lista fișierelor din subfolderul ales (ignorăm eventualele directoare ascunse)
    fisiere = [f for f in os.listdir(cale_subfolder)
               if os.path.isfile(os.path.join(cale_subfolder, f))]

    if not fisiere:
        print('Atenție: Subfolderul ',eticheta_aleasa,'este gol.')
        return None, eticheta_aleasa

    # 2. Alegem în mod aleator un fișier din acel subfolder
    fisier_ales = random.choice(fisiere)
    cale_complet_fisier = os.path.join(cale_subfolder, fisier_ales)

    return cale_complet_fisier, eticheta_aleasa



# Main processing flow (signal --> feature extractor --> classification )
# Ignore first run (for delays)
Includes latency evaluation
One may select among several types of feature extractor (FE) - pretrained classifiers (.tflite model) are included  

In [ ]:
# run only at the beginning of experiments to reset counters
contor=0  # experiment counter
contor_gres=0 # error counter

# Run each time for a new signal sequence
# Select the desired feature extractor or the source of signal sequences

In [ ]:
import time as ti
from_folder=True  # if False take available wav files (you need to label them manually)
sig_pro='iRDTvB1' # 'MFCC13' or 'iRDTvA1' or 'iRDTvB1'  # signal processor (.tflite classifier was previously trained using it)


# ---------- Prepare signal processor (feature extractor and scaler + classifier)
# scaling values (us same values as usen when the training set vas generated )
if sig_pro=='MFCC13':
  xmi=  -848.0403; xma= 299.19104
  Mmax=32
  scala = 1
  Mmax=32
  model_pth='mfcc13_VRES3_Acc_91_85_sc1_91.04_float32.tflite'
elif sig_pro=='iRDTvA1':
  xma=354.9277  # scaling factor
  xmi=0
  scala = 32   # specific value recorded when the training set was generated
  Mmax=83      # M feature for 0-padding
  model_pth='iRDTv1_VRES3_Acc_93_65_sc_32_float32.tflite'
elif sig_pro=='iRDTvB1':
  xma=355.39423  # scaling factor
  xmi=0
  scala = 8   # specific value recorded when the training set was generated
  Mmax=83      # M feature for 0-padding
  model_pth='iRDTv-op_VRES3_Acc_94_75_sc_8_float32.tflite'

#  ----------------------- GET SIGNAL -----------------------------------------------------
if from_folder:
  # Take a new signal sequence from folder
  fisier, eticheta = extrage_fisier_aleator('KWS')
else:
  # Or select a local wav with known label (eticheta)
  # labels are:  0Yes, 1No, 2Down, 3Up, 4Left, 5Right, 6Off, 7On, 8Go, 9Stop, xSilent, yUnknown
  fisier = '/content/yes1.wav' ; eticheta = '0yes'


# ============== START PROCESSING ==============================================
SAMPLE_RATE=16000
# read signal in -1,1 format
signal, sample_rate = librosa.load(fisier, sr=SAMPLE_RATE)

# ========================= apply feature extractor  =========================================
Nt=100 # loops with Nt=100 to get a better time estimate
if sig_pro=='MFCC13':
  t1=ti.time()
  for _ in range(Nt):
    feat2d=librosa.feature.mfcc(y=signal, sr=SAMPLE_RATE, n_mfcc=13, n_fft=2048, hop_length=512).T
  t2=ti.time()
elif sig_pro=='iRDTvA1':
  t1=ti.time()
  for _ in range(Nt):
    (fer, feat2d)=irdtv(signal, M=3, w=64, chan=[1,2,4,8,16,32], biti=24)
  t2=ti.time()
elif sig_pro=='iRDTvB1':
  t1=ti.time()
  for _ in range(Nt):
    (fer, feat2d)=irdtv(signal, M=3, w=64, chan=[1, 2, 4, 6, 8, 12, 16, 24, 32], biti=24)
  t2=ti.time()
t2=ti.time(); proc_time=((t2-t1)/100)

# 0-padding applied to feature matrix
Mcur=feat2d.shape[0]
x_i=np.pad(feat2d, pad_width=((0,Mmax-Mcur),(0,0)), mode='constant', constant_values=0)

# ========================= SCALER =============================================
xx_inp= scala*(x_i.astype('float32')-xmi)/(xma-xmi)
# Reshape xx_inp to match the model's expected input shape
xx_inp_reshaped = xx_inp[np.newaxis, :, :, np.newaxis]

#======================= invoke CLASSIFIER (.tflite) ===========================
interpreter = Interpreter(model_path=model_pth)
interpreter.allocate_tensors()
input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()

t1=ti.time()
interpreter.set_tensor(input_details[0]['index'], xx_inp_reshaped)
interpreter.invoke()
pred = interpreter.get_tensor(output_details[0]['index'])
t2=ti.time()
# Decision =====================================================================
clasa_ghicita = np.argmax(pred)
# For whatever reason labels were scrambled (index swap)
# in this dataset when compared to the one used for training
# solved by re-alignment
align={0:0, 1:11, 2:2, 3:3, 4:4, 5:5, 6:9, 7:10, 8:8, 9:1, 10:7,11:6}
clasa_ghici=align[clasa_ghicita]

if clasa_ghici==11:
  clasa_ghici='y'
if clasa_ghici==10:
  clasa_ghici='x'
else:
  clasa_ghici=str(clasa_ghici)
print('Real class : ', eticheta)
print('Predicted class: ',clasa_ghici)
contor+=1

if clasa_ghici[0]==eticheta[0]:
  print('Correct')
else:
  print('False'); contor_gres+=1
print(sig_pro+' feature extractor latency: ',np.round((proc_time)*1000,4),'milli-seconds')
print('classifier latency: ',np.round((t2-t1)*1000,2),'milli-seconds')
print(contor,'experiments: ')
print(contor_gres,'in', contor, 'failed: test_accuracy=',100*(contor-contor_gres)/contor,'%')
from IPython.display import Audio


Real class :  yUnknown
Predicted class:  y
Correct
iRDTvB1 feature extractor latency:  0.2653 milli-seconds
classifier latency:  1.15 milli-seconds
10 experiments: 
0 in 10 failed: test_accuracy= 100.0 %


In [ ]:
print('play the sound')
Audio(data=signal, rate=SAMPLE_RATE)

play the sound
